# AML 금융보안 챗봇 구현



## 0. 환경 설정

In [ ]:
import os
import sys
import json
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv()

print(f"✓ 프로젝트 루트: {project_root}")
print(f"✓ OPENAI_API_KEY: {'설정됨' if os.getenv('OPENAI_API_KEY') else '미설정'}")

## 1. State 기반 거래 데이터 모델

In [ ]:
from typing_extensions import TypedDict
from datetime import datetime

# 거래 데이터 모델
class TransactionData(TypedDict):
    """거래 정보"""
    transaction_id: str
    amount: float
    currency: str
    type: str  # wire_transfer, crypto, investment, normal_transfer
    channel: str  # online, branch, atm, crypto
    source_country: str
    dest_country: str
    frequency: int  # 24시간 내 거래 횟수
    timestamp: str

# 챗봇 시스템 상태
class ChatbotState(TypedDict):
    """챗봇 시스템 상태"""
    user_query: str
    transaction: dict
    routing_decision: str
    analysis_results: dict
    response: str
    conversation_history: list

print("✓ State 모델 정의 완료")

## 2. 샘플 거래 데이터

In [ ]:
from datetime import datetime

# 샘플 거래 데이터 (5가지 시나리오)
sample_transactions = [
    {
        "transaction_id": "TXN-001",
        "amount": 50000,
        "currency": "USD",
        "type": "wire_transfer",
        "channel": "online",
        "source_country": "KR",
        "dest_country": "US",
        "frequency": 3,
        "timestamp": "2024-01-15 10:30:00"
    },
    {
        "transaction_id": "TXN-002",
        "amount": 500000,
        "currency": "USD",
        "type": "wire_transfer",
        "channel": "online",
        "source_country": "RU",
        "dest_country": "US",
        "frequency": 1,
        "timestamp": "2024-01-15 11:15:00"
    },
    {
        "transaction_id": "TXN-003",
        "amount": 100000,
        "currency": "USD",
        "type": "crypto_exchange",
        "channel": "crypto",
        "source_country": "KR",
        "dest_country": "KR",
        "frequency": 25,
        "timestamp": "2024-01-15 12:00:00"
    },
    {
        "transaction_id": "TXN-004",
        "amount": 10000,
        "currency": "USD",
        "type": "investment",
        "channel": "online",
        "source_country": "US",
        "dest_country": "SG",
        "frequency": 5,
        "timestamp": "2024-01-15 13:20:00"
    },
    {
        "transaction_id": "TXN-005",
        "amount": 2000,
        "currency": "USD",
        "type": "normal_transfer",
        "channel": "atm",
        "source_country": "KR",
        "dest_country": "KR",
        "frequency": 1,
        "timestamp": "2024-01-15 14:45:00"
    }
]

# 거래 유형별 설명
tx_descriptions = {
    "wire_transfer": "전신송금 (해외 송금)",
    "crypto_exchange": "암호화폐 거래소 거래",
    "investment": "투자 거래",
    "normal_transfer": "일반 송금"
}

print(f"✓ 샘플 거래 {len(sample_transactions)}개 생성")
for tx in sample_transactions:
    print(f"  - {tx['transaction_id']}: {tx['amount']:,} {tx['currency']} ({tx_descriptions[tx['type']]})")

## 3. LLM 체인을 이용한 분석

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# LLM 초기화
api_key = os.getenv("OPENAI_API_KEY")
llm = ChatOpenAI(model="gpt-4o-mini", api_key=api_key, temperature=0.7)
analyzer_llm = ChatOpenAI(model="gpt-4o-mini", api_key=api_key, temperature=0)

print("✓ LLM 초기화 완료")

## 4. Node 함수들 - 거래 분석

In [ ]:
# Node 1: 일반 질문 처리
def general_chat(state: ChatbotState):
    """일반 질문에 대한 답변"""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "당신은 친절한 금융보안 상담사입니다. 사용자의 질문에 도움이 되는 답변을 제공하세요."),
        ("human", "{query}")
    ])
    
    chain = prompt | llm | StrOutputParser()
    response = chain.invoke({"query": state["user_query"]})
    
    return {"response": response}

# 테스트
test_state = {
    "user_query": "자금세탁(AML)이란 무엇인가요?",
    "transaction": {},
    "routing_decision": "",
    "analysis_results": {},
    "response": "",
    "conversation_history": []
}

print("[Node 1: 일반 질문]")
print(f"Q: {test_state['user_query']}")
result = general_chat(test_state)
print(f"A: {result['response'][:150]}...\n")

In [ ]:
# Node 2: 전신송금 거래 분석
def wire_transfer_analysis(state: ChatbotState):
    """전신송금 거래 분석"""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "당신은 AML 전문가입니다. 전신송금 거래의 위험도를 분석하세요."),
        ("human", "거래: {transaction}\n질문: {query}")
    ])
    
    chain = prompt | llm | StrOutputParser()
    response = chain.invoke({
        "transaction": json.dumps(state["transaction"], indent=2, ensure_ascii=False),
        "query": state["user_query"]
    })
    
    return {"response": response, "analysis_results": {"type": "wire_transfer"}}

# 테스트
test_state2 = {
    "user_query": "이 거래는 안전한가요?",
    "transaction": sample_transactions[0],
    "routing_decision": "",
    "analysis_results": {},
    "response": "",
    "conversation_history": []
}

print("[Node 2: 전신송금 분석]")
print(f"거래: {test_state2['transaction']['transaction_id']} (${test_state2['transaction']['amount']:,})")
result = wire_transfer_analysis(test_state2)
print(f"분석 결과: {result['response'][:150]}...\n")

In [ ]:
# Node 3: 암호화폐 거래 분석
def crypto_analysis(state: ChatbotState):
    """암호화폐 거래 분석"""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "당신은 암호화폐 자금세탁 탐지 전문가입니다. 암호화폐 거래의 위험성을 평가하세요."),
        ("human", "거래: {transaction}\n질문: {query}")
    ])
    
    chain = prompt | llm | StrOutputParser()
    response = chain.invoke({
        "transaction": json.dumps(state["transaction"], indent=2, ensure_ascii=False),
        "query": state["user_query"]
    })
    
    return {"response": response, "analysis_results": {"type": "crypto"}}

# 테스트
test_state3 = {
    "user_query": "이 암호화폐 거래가 의심스럽나요?",
    "transaction": sample_transactions[2],
    "routing_decision": "",
    "analysis_results": {},
    "response": "",
    "conversation_history": []
}

print("[Node 3: 암호화폐 분석]")
print(f"거래: {test_state3['transaction']['transaction_id']} (${test_state3['transaction']['amount']:,})")
print(f"빈도: {test_state3['transaction']['frequency']}회/24h")
result = crypto_analysis(test_state3)
print(f"분석 결과: {result['response'][:150]}...\n")

In [ ]:
# Node 4: 고액 거래 집중 검토
def high_amount_scrutiny(state: ChatbotState):
    """고액 거래 집중 검토"""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "당신은 고액거래 감시 전문가입니다. 고액 거래는 자금세탁 위험이 높습니다. 특별히 주의깊게 검토하세요."),
        ("human", "고액거래 정보: {transaction}\n질문: {query}\n\n고액 거래에 대한 집중 검토 의견을 제시하세요.")
    ])
    
    chain = prompt | llm | StrOutputParser()
    response = chain.invoke({
        "transaction": json.dumps(state["transaction"], indent=2, ensure_ascii=False),
        "query": state["user_query"]
    })
    
    return {"response": response, "analysis_results": {"alert": "high_amount"}}

# 테스트 - 고액 거래
test_state4 = {
    "user_query": "이 고액 거래를 승인해야 하나요?",
    "transaction": sample_transactions[1],  # $500,000
    "routing_decision": "",
    "analysis_results": {},
    "response": "",
    "conversation_history": []
}

print("[Node 4: 고액 거래 검토]")
print(f"거래: {test_state4['transaction']['transaction_id']} (${test_state4['transaction']['amount']:,})")
print(f"출발국: {test_state4['transaction']['source_country']}")
result = high_amount_scrutiny(test_state4)
print(f"검토 결과: {result['response'][:150]}...\n")

In [ ]:
# Node 5: 빈번 거래 경고
def frequent_transaction_alert(state: ChatbotState):
    """빈번 거래 경고"""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "당신은 거래 패턴 분석 전문가입니다. 빈번한 거래는 Layering 자금세탁의 신호입니다."),
        ("human", "빈번 거래 정보: {transaction}\n질문: {query}\n\n이 거래 패턴의 위험성을 평가하세요.")
    ])
    
    chain = prompt | llm | StrOutputParser()
    response = chain.invoke({
        "transaction": json.dumps(state["transaction"], indent=2, ensure_ascii=False),
        "query": state["user_query"]
    })
    
    return {"response": response, "analysis_results": {"alert": "frequent_transaction"}}

# 테스트 - 빈번한 거래
test_state5 = {
    "user_query": "이렇게 자주 거래를 해도 되나요?",
    "transaction": sample_transactions[2],  # 25회/24h
    "routing_decision": "",
    "analysis_results": {},
    "response": "",
    "conversation_history": []
}

print("[Node 5: 빈번 거래 경고]")
print(f"거래: {test_state5['transaction']['transaction_id']}")
print(f"빈도: {test_state5['transaction']['frequency']}회/24h (임계값: 10회)")
result = frequent_transaction_alert(test_state5)
print(f"경고: {result['response'][:150]}...\n")

## 5. LangGraph 그래프 구축

In [ ]:
from src.aml_chatbot import build_chatbot_graph

# 그래프 빌드
graph = build_chatbot_graph()

print("✓ LangGraph 그래프 구축 완료")
print(f"✓ 그래프 타입: {type(graph)}")
print(f"✓ 컴파일된 그래프: {graph}")

## 6. 거래별 분석 시나리오

In [ ]:
from src.aml_chatbot import chat

print("="*70)
print("거래별 분석 시나리오")
print("="*70)

# 시나리오 1: 일반 질문
print("\n[시나리오 1] 일반 질문")
query1 = "자금세탁의 3단계를 설명해줄 수 있나요?"
print(f"Q: {query1}")
response1 = chat(query1)
print(f"A: {response1}\n")

# 시나리오 2: 정상 전신송금
print("[시나리오 2] 정상 전신송금")
tx2 = sample_transactions[0]
query2 = "이 거래의 위험도를 평가해주세요."
print(f"Q: ${tx2['amount']:,} {tx2['type']} ({tx2['source_country']} → {tx2['dest_country']})")
response2 = chat(query2, tx2)
print(f"A: {response2}\n")

# 시나리오 3: 고액 + 고위험국
print("[시나리오 3] 고액 + 고위험국 (위험 거래)")
tx3 = sample_transactions[1]
query3 = "이 거래를 승인할 수 있을까요?"
print(f"Q: ${tx3['amount']:,} {tx3['type']} ({tx3['source_country']} → {tx3['dest_country']})")
response3 = chat(query3, tx3)
print(f"A: {response3}\n")

# 시나리오 4: 빈번한 암호화폐 거래
print("[시나리오 4] 빈번한 암호화폐 거래 (Layering 신호)")
tx4 = sample_transactions[2]
query4 = "이 거래 패턴이 정상인가요?"
print(f"Q: {tx4['frequency']}회/24h {tx4['type']} (채널: {tx4['channel']})")
response4 = chat(query4, tx4)
print(f"A: {response4}\n")

## 7. Conditional Edge 통합 시연

In [ ]:
from src.aml_chatbot import (
    route_by_transaction_type,
    route_by_amount,
    route_by_frequency,
    route_by_risk_country,
    route_by_channel
)

print("="*70)
print("Conditional Edge 동작 확인")
print("="*70)

# 고액 + 고위험국 거래 분석
tx = sample_transactions[1]  # $500,000 from RU to US
state = {
    "user_query": "이 거래 위험도는?",
    "transaction": tx,
    "routing_decision": "",
    "analysis_results": {},
    "response": "",
    "conversation_history": []
}

print(f"\n거래: {tx['transaction_id']}")
print(f"  - 거래액: ${tx['amount']:,}")
print(f"  - 거래 유형: {tx['type']}")
print(f"  - 출발국: {tx['source_country']} → 도착국: {tx['dest_country']}")
print(f"  - 채널: {tx['channel']}")
print(f"  - 빈도: {tx['frequency']}회/24h")

print(f"\n라우팅 결정:")
print(f"  [Edge 1] 거래 유형: {route_by_transaction_type(state)}")
print(f"  [Edge 2] 거래액: {route_by_amount(state)}")
print(f"  [Edge 3] 거래 빈도: {route_by_frequency(state)}")
print(f"  [Edge 4] 위험 국가: {route_by_risk_country(state)}")
print(f"  [Edge 5] 채널: {route_by_channel(state)}")

print(f"\n➜ 결론: 5개 모든 Conditional Edge가 위험 신호 감지!")

## 8. 학습 내용 정리

### AML 금융보안 챗봇 구현 요소

| 요소 | 설명 | 구현 |
|------|------|------|
| **State** | 거래 정보 + 시스템 상태 | ChatbotState (TypedDict) |
| **Node 함수** | 각 거래 유형별 분석 | wire_transfer_analysis, crypto_analysis 등 |
| **LLM 체인** | ChatPromptTemplate + LLM + Parser | 거래 위험도 판정 |
| **Conditional Edge** | 5개 조건부 라우팅 | transaction_type, amount, frequency, risk_country, channel |
| **Graph** | 전체 워크플로우 | StateGraph + add_conditional_edges |

### 자금세탁 3단계 탐지

1. **Placement (입금)**: 불법 자금의 금융 시스템 진입
   - 신호: 새로운 계좌에서 대량 입금
   - 탐지: 거래액 + 거래 빈도 분석

2. **Layering (은폐)**: 여러 거래로 자금 출처 불명확화
   - 신호: 빈번한 거래, 암호화폐 환전
   - 탐지: 거래 빈도 + 채널 분석

3. **Integration (통합)**: 세탁된 자금을 합법적으로 사용
   - 신호: 고위험국과의 거래, 투자 활동
   - 탐지: 위험국가 + 거래 유형 분석

